# P4: Full MLOps Pipeline

**Module 3 — Week 13**

## Problem Statement
Build an end-to-end MLOps workflow: experiment tracking with MLflow, model registry lifecycle management, A/B testing framework, data drift detection, and retraining trigger logic.

## Success Metrics
- Best model AUC > 0.80
- A/B test reaches statistical significance
- Drift detected when present

---
## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../../..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
    mlflow.set_tracking_uri("./mlruns")
    print(f"MLflow {mlflow.__version__} available")
except ImportError:
    MLFLOW_AVAILABLE = False
    print("MLflow not installed — training will proceed without tracking")

from src.train import generate_churn_data, prepare_features, train_model, train_all_models
from src.ab_test import ABTestSimulator

In [ ]:
# Verify data generation
df = generate_churn_data()
print(f"Dataset shape: {df.shape}")
print(f"Churn rate: {df['churned'].mean():.3f}")
df.head()

---
## 2. Train Multiple Models with MLflow

In [ ]:
# TODO: Train all models using train_all_models()
# Store: results, best_name, best_model, data splits
# Display a summary table of all model metrics

# --- YOUR CODE HERE ---


# --- SOLUTION (uncomment to use) ---
# results, best_name, best_model, (X_train, X_test, y_train, y_test) = train_all_models()
#
# # Build summary table
# summary = pd.DataFrame({
#     name: info['metrics'] for name, info in results.items()
# }).T
# summary.index.name = 'Model'
# print("\nModel Comparison:")
# display(summary.sort_values('auc', ascending=False))

---
## 3. Query and Compare Runs

In [ ]:
# TODO: If MLflow is available, use mlflow.search_runs() to query the experiment
# and display results as a DataFrame sorted by AUC.
# If MLflow is not available, use the results dict from Section 2.
# Then plot a bar chart comparing accuracy, F1, and AUC across models.

# --- YOUR CODE HERE ---


# --- SOLUTION (uncomment to use) ---
# if MLFLOW_AVAILABLE:
#     runs_df = mlflow.search_runs(experiment_names=["churn-experiment"])
#     cols = ['run_id', 'tags.mlflow.runName', 'metrics.accuracy', 'metrics.f1', 'metrics.auc']
#     display(runs_df[cols].sort_values('metrics.auc', ascending=False))
# else:
#     print("MLflow not available — using results dict")
#
# # Plot metric comparison
# metrics_df = pd.DataFrame({name: info['metrics'] for name, info in results.items()}).T
# metrics_df.plot(kind='bar', figsize=(10, 5), rot=0)
# plt.title('Model Comparison')
# plt.ylabel('Score')
# plt.ylim(0.5, 1.0)
# plt.legend(loc='lower right')
# plt.tight_layout()
# plt.show()

---
## 4. Register Best Model

In [ ]:
# TODO: If MLflow is available:
#   1. Register the best model in the Model Registry
#   2. Transition through lifecycle: None → Staging → Production
#   3. Print confirmation at each stage
# If MLflow is not available, print what would happen.

# --- YOUR CODE HERE ---


# --- SOLUTION (uncomment to use) ---
# model_name_registry = "churn-predictor"
#
# if MLFLOW_AVAILABLE:
#     # Find best run
#     runs_df = mlflow.search_runs(experiment_names=["churn-experiment"],
#                                   order_by=["metrics.auc DESC"])
#     best_run_id = runs_df.iloc[0]['run_id']
#     model_uri = f"runs:/{best_run_id}/model"
#
#     # Register
#     mv = mlflow.register_model(model_uri, model_name_registry)
#     print(f"Registered: {mv.name} v{mv.version}")
#
#     # Transition to Staging
#     client = mlflow.MlflowClient()
#     client.transition_model_version_stage(
#         name=model_name_registry, version=mv.version, stage="Staging"
#     )
#     print(f"Transitioned v{mv.version} → Staging")
#
#     # Transition to Production
#     client.transition_model_version_stage(
#         name=model_name_registry, version=mv.version, stage="Production"
#     )
#     print(f"Transitioned v{mv.version} → Production")
# else:
#     print(f"[Simulated] Would register {best_name} as '{model_name_registry}'")
#     print(f"[Simulated] Lifecycle: None → Staging → Production")

---
## 5. A/B Test Simulation

In [ ]:
# TODO: Create an ABTestSimulator instance
# Simulate 10,000 users: Model A churn rate = 0.25, Model B churn rate = 0.22
# Print a summary of the raw results (group sizes, churn counts)

# --- YOUR CODE HERE ---


# --- SOLUTION (uncomment to use) ---
# simulator = ABTestSimulator(seed=42)
# ab_results = simulator.simulate(
#     n_users=10000,
#     model_a_churn_rate=0.25,
#     model_b_churn_rate=0.22
# )
#
# print(f"Total users: {len(ab_results)}")
# print(f"\nGroup summary:")
# print(ab_results.groupby('group')['churned'].agg(['count', 'sum', 'mean']).round(4))
# print(f"\nSample size needed (3% effect): {simulator.required_sample_size(0.25, 0.03)}")

---
## 6. Statistical Analysis

In [ ]:
# TODO: Run analyzer.analyze() on the A/B results
# Print all metrics: churn rates, absolute/relative diff, p-value, effect size,
# significance, confidence intervals, business impact
# Interpret the results

# --- YOUR CODE HERE ---


# --- SOLUTION (uncomment to use) ---
# analysis = simulator.analyze(ab_results)
#
# print("=== A/B Test Results ===")
# print(f"\nChurn Rates:")
# print(f"  Model A (control):   {analysis['rate_a']:.4f}  CI: ({analysis['ci_a'][0]:.4f}, {analysis['ci_a'][1]:.4f})")
# print(f"  Model B (treatment): {analysis['rate_b']:.4f}  CI: ({analysis['ci_b'][0]:.4f}, {analysis['ci_b'][1]:.4f})")
# print(f"\nDifferences:")
# print(f"  Absolute: {analysis['absolute_diff']:.4f}")
# print(f"  Relative: {analysis['relative_diff']:.2%}")
# print(f"\nStatistical Test:")
# print(f"  Chi-squared: {analysis['chi2']:.4f}")
# print(f"  p-value: {analysis['p_value']:.6f}")
# print(f"  Significant (p<0.05): {analysis['significant']}")
# print(f"  Effect size (Cohen's h): {analysis['effect_size_h']:.4f}")
# print(f"\nBusiness Impact:")
# print(f"  Saved customers: {analysis['saved_customers']:.0f}")
# print(f"  Value saved: ${analysis['value_saved']:,.0f}")
# print(f"  Net value: ${analysis['net_value']:,.0f}")
#
# # Visualize
# simulator.visualize(ab_results, analysis)

---
## 7. Drift Detection

In [ ]:
# TODO:
# 1. Implement compute_psi(reference, current, bins=10) — Population Stability Index
# 2. Simulate drift: take original numeric features, create 4 time periods where
#    monthly_charges and tenure gradually shift
# 3. Compute PSI for each feature in each period
# 4. Plot PSI over time with threshold lines (0.1 = moderate, 0.25 = significant)
# 5. Run KS test on each feature across periods

# --- YOUR CODE HERE ---


# --- SOLUTION (uncomment to use) ---
# def compute_psi(reference, current, bins=10):
#     """Compute Population Stability Index."""
#     eps = 1e-4
#     breakpoints = np.linspace(min(reference.min(), current.min()),
#                               max(reference.max(), current.max()), bins + 1)
#     ref_counts = np.histogram(reference, bins=breakpoints)[0] / len(reference)
#     cur_counts = np.histogram(current, bins=breakpoints)[0] / len(current)
#     ref_counts = np.clip(ref_counts, eps, None)
#     cur_counts = np.clip(cur_counts, eps, None)
#     psi = np.sum((cur_counts - ref_counts) * np.log(cur_counts / ref_counts))
#     return psi
#
# # Reference data
# X_ref, _ = prepare_features(generate_churn_data(n=5000, seed=42))
# numeric_cols = ['tenure_months', 'monthly_charges', 'total_charges',
#                 'online_security', 'tech_support', 'senior_citizen', 'partner']
#
# # Simulate drift over 4 periods
# drift_results = {col: [] for col in numeric_cols}
# periods = ['T0 (baseline)', 'T1 (slight)', 'T2 (moderate)', 'T3 (severe)']
# shifts = [0, 0.5, 1.5, 3.0]  # multipliers for drift magnitude
#
# for shift in shifts:
#     df_drift = generate_churn_data(n=5000, seed=99)
#     X_drift, _ = prepare_features(df_drift)
#     # Apply drift to monthly_charges and tenure
#     if shift > 0:
#         X_drift['monthly_charges'] += shift * 8
#         X_drift['tenure_months'] -= shift * 3
#         X_drift['total_charges'] = X_drift['monthly_charges'] * X_drift['tenure_months'].clip(1)
#     for col in numeric_cols:
#         if col in X_drift.columns:
#             psi = compute_psi(X_ref[col].values, X_drift[col].values)
#             drift_results[col].append(psi)
#
# # Plot PSI over time
# fig, ax = plt.subplots(figsize=(12, 6))
# for col in numeric_cols:
#     if drift_results[col]:
#         ax.plot(periods, drift_results[col], marker='o', label=col)
# ax.axhline(y=0.1, color='orange', linestyle='--', label='Moderate drift (0.1)')
# ax.axhline(y=0.25, color='red', linestyle='--', label='Significant drift (0.25)')
# ax.set_ylabel('PSI')
# ax.set_title('Population Stability Index Over Time')
# ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.tight_layout()
# plt.show()
#
# # KS test for the severe drift period
# print("\nKS Test (Reference vs Severe Drift):")
# df_severe = generate_churn_data(n=5000, seed=99)
# X_severe, _ = prepare_features(df_severe)
# X_severe['monthly_charges'] += 3.0 * 8
# X_severe['tenure_months'] -= 3.0 * 3
# for col in numeric_cols:
#     if col in X_severe.columns:
#         ks_stat, ks_p = stats.ks_2samp(X_ref[col].values, X_severe[col].values)
#         flag = " *** DRIFT" if ks_p < 0.05 else ""
#         print(f"  {col:20s} KS={ks_stat:.4f}  p={ks_p:.6f}{flag}")

---
## 8. Retraining Logic

In [ ]:
# TODO: Implement should_retrain(psi_scores, auc_current, auc_baseline, days_since_training)
# with the following thresholds:
#   - Any PSI > 0.25 → retrain (significant drift)
#   - AUC drop > 5% from baseline → retrain
#   - Days since training > 90 → retrain (staleness)
#   - Average PSI > 0.1 → retrain (moderate overall drift)
# Test with various scenarios and print the decision + reason.

# --- YOUR CODE HERE ---


# --- SOLUTION (uncomment to use) ---
# def should_retrain(psi_scores, auc_current, auc_baseline, days_since_training):
#     """Decide whether to retrain based on monitoring signals."""
#     reasons = []
#
#     # Check significant drift on any feature
#     for feature, psi in psi_scores.items():
#         if psi > 0.25:
#             reasons.append(f"Significant drift on {feature} (PSI={psi:.3f})")
#
#     # Check average PSI
#     avg_psi = np.mean(list(psi_scores.values()))
#     if avg_psi > 0.1:
#         reasons.append(f"Moderate overall drift (avg PSI={avg_psi:.3f})")
#
#     # Check AUC degradation
#     auc_drop = (auc_baseline - auc_current) / auc_baseline
#     if auc_drop > 0.05:
#         reasons.append(f"AUC degraded by {auc_drop:.1%} ({auc_baseline:.4f} → {auc_current:.4f})")
#
#     # Check staleness
#     if days_since_training > 90:
#         reasons.append(f"Model is {days_since_training} days old (>90 day threshold)")
#
#     return len(reasons) > 0, reasons
#
# # Test scenarios
# scenarios = [
#     {"name": "Healthy model",
#      "psi": {"tenure": 0.02, "charges": 0.03}, "auc": 0.84, "baseline": 0.85, "days": 30},
#     {"name": "Significant drift",
#      "psi": {"tenure": 0.05, "charges": 0.30}, "auc": 0.83, "baseline": 0.85, "days": 45},
#     {"name": "Performance degraded",
#      "psi": {"tenure": 0.04, "charges": 0.06}, "auc": 0.78, "baseline": 0.85, "days": 60},
#     {"name": "Stale model",
#      "psi": {"tenure": 0.03, "charges": 0.04}, "auc": 0.84, "baseline": 0.85, "days": 120},
#     {"name": "Multiple triggers",
#      "psi": {"tenure": 0.15, "charges": 0.35}, "auc": 0.75, "baseline": 0.85, "days": 100},
# ]
#
# for s in scenarios:
#     retrain, reasons = should_retrain(s['psi'], s['auc'], s['baseline'], s['days'])
#     status = "RETRAIN" if retrain else "OK"
#     print(f"\n[{status}] {s['name']}")
#     for r in reasons:
#         print(f"  - {r}")

---
## 9. What I Built / What I Learned

### Summary
- TODO: Describe the end-to-end MLOps pipeline you built

### Key Learnings
- TODO: What did you learn about MLflow experiment tracking?
- TODO: What did you learn about A/B test design and analysis?
- TODO: What did you learn about drift detection (PSI, KS test)?
- TODO: How would you implement retraining triggers in a production system?

### What I Would Do Differently in Production
- TODO: What additional monitoring would you add?
- TODO: How would you automate the retraining pipeline?
- TODO: What CI/CD considerations are important for ML models?